# Amazon — EmbeddingANN (v2 — Overfitting Fixed)

**Problem with v1**: Val RMSE exploded from 1.86 → 7.5 over 20 epochs.
Root cause — embedding tables were too large relative to dataset size.

**Fixes applied:**
| Fix | Detail |
|-----|--------|
| Filter active users/products | Only embed users/products with ≥ 5 reviews. Rest → global avg |
| Smaller embedding dims | 8 instead of 32 — fewer parameters |
| Embedding dropout | 0.5 dropout after each embedding lookup |
| L2 regularisation | weight_decay=1e-5 on Adam |
| Early stopping | Stop when val RMSE stops improving |

**Reads from**: `data/train_features.csv`
**Saves to**  : `embeddings/`


## Roadmap
```
STEP 1  — Imports & paths
STEP 2  — Load data
STEP 3  — Filter to active users & products (≥5 reviews)
STEP 4  — Label encode filtered IDs
STEP 5  — Scale numerical features
STEP 6  — PyTorch Dataset & DataLoader
STEP 7  — Model definition (EmbeddingANN v2)
STEP 8  — Train with early stopping
STEP 9  — Evaluate
STEP 10 — Save embeddings to disk
```

---
## Step 1 — Imports & Paths

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device: {DEVICE}")


In [ ]:
DATA_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\data"
EMB_DIR  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\embeddings"
os.makedirs(EMB_DIR, exist_ok=True)
print(f"Data dir : {DATA_DIR}")
print(f"Emb dir  : {EMB_DIR}")


---
## Step 2 — Load Data

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_features.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_features.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_features.csv'))

FEATURE_COLS = [
    'user_avg_rating', 'user_rating_count',
    'product_avg_rating', 'product_rating_count', 'product_rating_std',
    'days_since_review'
]
TARGET = 'ratings'

print(f"Train : {df_train.shape}")
print(f"Val   : {df_val.shape}")
print(f"Test  : {df_test.shape}")


---
## Step 3 — Filter to Active Users & Products

**Why filter?**
With 400K+ unique users and 200K+ products, embedding tables would have
millions of parameters — the model memorises specific users/products
rather than learning generalisable patterns.

**Strategy:**
- Users/products with ≥ MIN_REVIEWS reviews → get a learned embedding
- Users/products below threshold → assigned index 0 (<UNK>) → zero vector
  (their signal comes entirely from the 6 numerical features)

This dramatically reduces vocab size while keeping most of the signal
since low-activity users/products have little rating history anyway.


In [ ]:
MIN_REVIEWS = 5   # minimum reviews to get a dedicated embedding

# Count reviews per user and product in train
user_counts    = df_train['userId'].value_counts()
product_counts = df_train['productId'].value_counts()

active_users    = set(user_counts[user_counts    >= MIN_REVIEWS].index)
active_products = set(product_counts[product_counts >= MIN_REVIEWS].index)

print(f"Total unique users    : {df_train['userId'].nunique():,}")
print(f"Active users (>={MIN_REVIEWS} reviews): {len(active_users):,}  "
      f"({len(active_users)/df_train['userId'].nunique()*100:.1f}%)")
print(f"Total unique products : {df_train['productId'].nunique():,}")
print(f"Active products(>={MIN_REVIEWS} reviews): {len(active_products):,}  "
      f"({len(active_products)/df_train['productId'].nunique()*100:.1f}%)")


---
## Step 4 — Label Encode Filtered IDs

Only active users/products get a unique index.
All others → index 0 (<UNK>) → zero embedding vector.
Their rating prediction relies entirely on the 6 numerical features.


In [ ]:
# Build vocab from active users/products only
user_vocab    = ['<UNK>'] + sorted(active_users)
product_vocab = ['<UNK>'] + sorted(active_products)

user2idx    = {u: i for i, u in enumerate(user_vocab)}
product2idx = {p: i for i, p in enumerate(product_vocab)}

n_users    = len(user2idx)
n_products = len(product2idx)

print(f"User vocab size    : {n_users:,}  (was {df_train['userId'].nunique():,})")
print(f"Product vocab size : {n_products:,}  (was {df_train['productId'].nunique():,})")


In [ ]:
def encode_ids(df, user2idx, product2idx):
    """Map IDs to indices. Inactive/unseen IDs → 0 (<UNK>)."""
    user_idx = df['userId'].map(
        lambda x: user2idx.get(x, 0)).values.astype(np.int64)
    prod_idx = df['productId'].map(
        lambda x: product2idx.get(x, 0)).values.astype(np.int64)
    return user_idx, prod_idx

train_user_idx, train_prod_idx = encode_ids(df_train, user2idx, product2idx)
val_user_idx,   val_prod_idx   = encode_ids(df_val,   user2idx, product2idx)
test_user_idx,  test_prod_idx  = encode_ids(df_test,  user2idx, product2idx)

# Verify no out-of-bounds
assert train_user_idx.max() < n_users
assert train_prod_idx.max() < n_products
print("All indices valid ✓")

# Show what fraction gets a real embedding vs UNK
unk_users = (train_user_idx == 0).mean() * 100
unk_prods = (train_prod_idx == 0).mean() * 100
print(f"Train rows with UNK user    : {unk_users:.1f}%")
print(f"Train rows with UNK product : {unk_prods:.1f}%")


In [ ]:
# Save encoders for classical model notebooks
with open(os.path.join(EMB_DIR, 'user_encoder.json'), 'w') as f:
    json.dump(user2idx, f)
with open(os.path.join(EMB_DIR, 'product_encoder.json'), 'w') as f:
    json.dump(product2idx, f)
print("Encoders saved.")


---
## Step 5 — Scale Numerical Features

In [ ]:
scaler      = StandardScaler()
X_train_num = scaler.fit_transform(df_train[FEATURE_COLS])
X_val_num   = scaler.transform(df_val[FEATURE_COLS])
X_test_num  = scaler.transform(df_test[FEATURE_COLS])

y_train = df_train[TARGET].values.astype(np.float32)
y_val   = df_val[TARGET].values.astype(np.float32)
y_test  = df_test[TARGET].values.astype(np.float32)

print(f"Numerical matrix — train: {X_train_num.shape}")


---
## Step 6 — PyTorch Dataset & DataLoader

In [ ]:
class AmazonDataset(Dataset):
    def __init__(self, num_arr, user_idx, prod_idx, labels):
        self.num      = torch.tensor(num_arr,  dtype=torch.float32)
        self.user_idx = torch.tensor(user_idx, dtype=torch.long)
        self.prod_idx = torch.tensor(prod_idx, dtype=torch.long)
        self.labels   = torch.tensor(labels,   dtype=torch.float32)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return (self.num[idx], self.user_idx[idx],
                self.prod_idx[idx], self.labels[idx])


In [ ]:
BATCH_SIZE = 512

train_loader = DataLoader(
    AmazonDataset(X_train_num, train_user_idx, train_prod_idx, y_train),
    batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(
    AmazonDataset(X_val_num, val_user_idx, val_prod_idx, y_val),
    batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(
    AmazonDataset(X_test_num, test_user_idx, test_prod_idx, y_test),
    batch_size=BATCH_SIZE, shuffle=False)

num_b, u_b, p_b, y_b = next(iter(train_loader))
print(f"Batch — num: {num_b.shape}  user: {u_b.shape}  prod: {p_b.shape}")


---
## Step 7 — Model Definition (EmbeddingANN v2)

**Changes vs v1:**
- Embedding dims reduced: 32 → 8 (fewer parameters)
- Dropout after each embedding lookup (0.5)
- Smaller MLP: 128→64 → 64→32 (appropriate for smaller input)

```
userId    → Embedding(n_users,    8) → Dropout(0.5) ──┐
productId → Embedding(n_products, 8) → Dropout(0.5) ──┤
6 features (scaled)                                 ──┤
                                                       ↓
                              concat [8+8+6 = 22]
                              Linear(22→64) → ReLU → Dropout(0.3)
                              Linear(64→32) → ReLU → Dropout(0.3)
                              Linear(32→1)
```


In [ ]:
class EmbeddingANN(nn.Module):
    def __init__(self, n_users, n_products, n_num,
                 user_emb_dim=8, prod_emb_dim=8,
                 emb_dropout=0.5, mlp_dropout=0.3):
        super().__init__()

        # Smaller embedding dims — reduces memorisation
        self.user_emb = nn.Embedding(n_users,    user_emb_dim, padding_idx=0)
        self.prod_emb = nn.Embedding(n_products, prod_emb_dim, padding_idx=0)

        # Dropout on embeddings — key regularisation for large vocab
        self.emb_dropout = nn.Dropout(emb_dropout)

        self.n_users    = n_users
        self.n_products = n_products

        mlp_input = user_emb_dim + prod_emb_dim + n_num
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input, 64), nn.ReLU(), nn.Dropout(mlp_dropout),
            nn.Linear(64, 32),        nn.ReLU(), nn.Dropout(mlp_dropout),
            nn.Linear(32, 1)
        )

    def forward(self, num, user_idx, prod_idx):
        user_idx = user_idx.clamp(0, self.n_users    - 1)
        prod_idx = prod_idx.clamp(0, self.n_products - 1)

        user_vec = self.emb_dropout(self.user_emb(user_idx))
        prod_vec = self.emb_dropout(self.prod_emb(prod_idx))

        x = torch.cat([user_vec, prod_vec, num], dim=1)
        return self.mlp(x).squeeze(1)


model = EmbeddingANN(n_users, n_products, len(FEATURE_COLS)).to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal parameters: {total:,}")


---
## Step 8 — Train with Early Stopping

Early stopping monitors val RMSE and stops training when it stops
improving for `patience` consecutive epochs.
This prevents the val RMSE explosion we saw in v1.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    model.train() if optimizer else model.eval()
    all_preds, all_labels = [], []
    ctx = torch.enable_grad() if optimizer else torch.no_grad()
    with ctx:
        for num_b, u_b, p_b, y_b in loader:
            num_b = num_b.to(DEVICE)
            u_b   = u_b.to(DEVICE)
            p_b   = p_b.to(DEVICE)
            y_b   = y_b.to(DEVICE)
            preds = model(num_b, u_b, p_b)
            loss  = criterion(preds, y_b)
            if optimizer:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(y_b.cpu().numpy())
    p = np.array(all_preds); l = np.array(all_labels)
    return (round(float(np.sqrt(mean_squared_error(l, p))), 4),
            round(float(mean_absolute_error(l, p)), 4))


In [ ]:
NUM_EPOCHS  = 30
PATIENCE    = 5      # stop if val RMSE doesn't improve for 5 epochs
LR          = 0.001
WEIGHT_DECAY= 1e-5   # L2 regularisation on all parameters including embeddings

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = {'train_rmse': [], 'val_rmse': []}

# Early stopping state
best_val_rmse  = float('inf')
best_epoch     = 0
patience_count = 0
best_state     = None

t0 = time.perf_counter()
print(f"Training for up to {NUM_EPOCHS} epochs (patience={PATIENCE})...")
print("-" * 60)

for epoch in range(NUM_EPOCHS):
    tr_rmse, _  = run_epoch(model, train_loader, criterion, optimizer)
    val_rmse, _ = run_epoch(model, val_loader,   criterion)

    history['train_rmse'].append(tr_rmse)
    history['val_rmse'].append(val_rmse)

    # Check for improvement
    if val_rmse < best_val_rmse:
        best_val_rmse  = val_rmse
        best_epoch     = epoch + 1
        patience_count = 0
        # Save best model weights
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_count += 1

    if (epoch + 1) % 5 == 0 or epoch == 0 or patience_count == PATIENCE:
        print(f"Epoch {epoch+1:>2}/{NUM_EPOCHS}  "
              f"Train RMSE: {tr_rmse}  Val RMSE: {val_rmse}  "
              f"{'← best' if patience_count == 0 else f'patience {patience_count}/{PATIENCE}'}")

    if patience_count >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

# Restore best weights
model.load_state_dict(best_state)
train_time = time.perf_counter() - t0
print(f"\nBest val RMSE: {best_val_rmse} at epoch {best_epoch}")
print(f"Total training time: {train_time:.1f}s")


In [ ]:
# Learning curves
epochs_run = len(history['train_rmse'])
plt.figure(figsize=(8, 3))
plt.plot(history['train_rmse'], label='Train RMSE')
plt.plot(history['val_rmse'],   label='Val RMSE')
plt.axvline(best_epoch - 1, color='red', linestyle='--',
            label=f'Best epoch ({best_epoch})')
plt.xlabel('Epoch'); plt.ylabel('RMSE')
plt.title('EmbeddingANN v2 — Learning Curves')
plt.legend(); plt.tight_layout(); plt.show()


---
## Step 9 — Evaluate on All Splits

In [ ]:
tr_rmse,  tr_mae  = run_epoch(model, train_loader, criterion)
val_rmse, val_mae = run_epoch(model, val_loader,   criterion)
te_rmse,  te_mae  = run_epoch(model, test_loader,  criterion)

print("=" * 55)
print("EmbeddingANN v2 — Final Results")
print("=" * 55)
print(f"  Train      RMSE: {tr_rmse}   MAE: {tr_mae}")
print(f"  Validation RMSE: {val_rmse}   MAE: {val_mae}")
print(f"  Test       RMSE: {te_rmse}   MAE: {te_mae}")
print(f"  Train time : {train_time:.1f}s")
print(f"  Best epoch : {best_epoch}")
print(f"  Train-Test gap : {round(te_rmse - tr_rmse, 4)}")


---
## Step 10 — Save Embeddings to Disk

Save weight matrices + encoders + results JSON.
Classical model notebooks load these for Experiment 2.


In [ ]:
# Extract weight matrices from best model
user_weights = model.user_emb.weight.detach().cpu().numpy()
prod_weights = model.prod_emb.weight.detach().cpu().numpy()

print(f"User embedding matrix    : {user_weights.shape}")
print(f"Product embedding matrix : {prod_weights.shape}")


In [ ]:
# Save weights
np.save(os.path.join(EMB_DIR, 'user_embeddings.npy'),    user_weights)
np.save(os.path.join(EMB_DIR, 'product_embeddings.npy'), prod_weights)

# Save results JSON
ann_results = {
    'model'        : 'EmbeddingANN (v2)',
    'train_rmse'   : tr_rmse,  'val_rmse'  : val_rmse,  'test_rmse'  : te_rmse,
    'train_mae'    : tr_mae,   'val_mae'   : val_mae,   'test_mae'   : te_mae,
    'train_time_s' : round(train_time, 1),
    'best_epoch'   : best_epoch,
    'config': {
        'user_emb_dim'  : 8,
        'prod_emb_dim'  : 8,
        'emb_dropout'   : 0.5,
        'mlp_dropout'   : 0.3,
        'weight_decay'  : 1e-5,
        'min_reviews'   : MIN_REVIEWS,
        'n_users'       : n_users,
        'n_products'    : n_products,
    }
}
with open(os.path.join(EMB_DIR, 'ann_results.json'), 'w') as f:
    json.dump(ann_results, f, indent=2)

# Confirm all files
expected = ['user_embeddings.npy', 'product_embeddings.npy',
            'user_encoder.json',   'product_encoder.json',
            'ann_results.json']
print("\nEmbedding folder:")
for fname in expected:
    fpath  = os.path.join(EMB_DIR, fname)
    status = '✓' if os.path.exists(fpath) else '✗ MISSING'
    size   = f"{os.path.getsize(fpath)/1e6:.1f} MB" if os.path.exists(fpath) else ""
    print(f"  {status}  {fname:<30}  {size}")
